In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import rasterio
import sys
import os 
from osgeo import ogr  
ogr.UseExceptions()
from datetime import datetime, timedelta
import warnings
import shapefile
import pandas as pd
import csv
from rasterio.crs import CRS
from shapely.geometry import shape
from shapely import wkt
import geopandas as gpd
from shapely.geometry import Point
from shapely.strtree import STRtree
from shapely.geometry import box
import psutil
import shutil
from pathlib import Path
import re


In [12]:
items = ['2025-10-11','2025-10-02','2025-11-02','2025-12-01']

[pd.to_datetime(x).tz_localize(None) for x in items if x]


[Timestamp('2025-10-11 00:00:00'),
 Timestamp('2025-10-02 00:00:00'),
 Timestamp('2025-11-02 00:00:00'),
 Timestamp('2025-12-01 00:00:00')]

In [11]:
items = pd.Series(items)
pd.to_datetime(items)

0   2025-10-11
1   2025-10-02
2   2025-11-02
3   2025-12-01
dtype: datetime64[us]

In [13]:
pd.to_datetime(items, errors="coerce").tz_localize(None)

DatetimeIndex(['2025-10-11', '2025-10-02', '2025-11-02', '2025-12-01'], dtype='datetime64[us]', freq=None)

In [3]:
df = pd.read_csv("data/new_model_training_dataset.csv")

In [4]:
df

,typhoon_name,typhoon_year,grid_point_id,wind_speed,track_distance,rainfall_max_6h,rainfall_max_24h,total_houses,rwi,strong_roof_strong_wall,...,std_tri,mean_elev,coast_length,with_coast,urban,rural,water,total_pop,percent_houses_damaged,percent_houses_damaged_5years
0,DURIAN,2006,101,0.0,303.180555,0.122917,0.085417,31.000000,NaN,22.580645,...,2.699781,5.762712,3445.709753,1,0.00,0.000000,1.000000,0.000000,0.0,0.000000
1,DURIAN,2006,4475,0.0,638.027502,0.091667,0.027083,3.301020,-0.527000,2.639401,...,4.585088,12.799127,8602.645832,1,0.00,0.000000,1.000000,0.000000,0.0,0.000000
2,DURIAN,2006,4639,0.0,603.631997,0.535417,0.146354,12.103741,-0.283000,2.639401,...,1.527495,8.833333,5084.012925,1,0.00,0.010000,0.990000,197.339034,0.0,0.000000
3,DURIAN,2006,4640,0.0,614.675270,0.356250,0.101562,645.899660,-0.358889,2.639401,...,11.677657,17.530431,55607.865950,1,0.00,0.310000,0.690000,4970.477311,0.0,0.000000
4,DURIAN,2006,4641,0.0,625.720905,0.202083,0.057812,1071.731293,-0.462800,2.639401,...,17.074011,31.931338,35529.342507,1,0.00,0.770000,0.230000,12408.594656,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141253,MOLAVE,2020,20677,0.0,644.575831,2.543750,0.778646,4449.357133,0.508167,44.762048,...,18.012771,36.304688,21559.003490,1,0.08,0.080000,0.840000,17619.701390,0.0,0.000000
141254,MOLAVE,2020,20678,0.0,655.685233,2.558333,0.861458,1521.435795,-0.174100,44.762048,...,13.163042,65.687266,12591.742022,1,0.00,0.420000,0.580000,5623.069564,0.0,0.000000
141255,MOLAVE,2020,20679,0.0,666.794635,2.975000,0.949479,930.647069,-0.244286,25.078318,...,10.901755,37.414996,19740.596834,1,0.00,0.109091,0.890909,5912.671746,0.0,0.015207
141256,MOLAVE,2020,20680,0.0,677.904037,2.889583,1.083333,1800.666044,0.038000,16.796996,...,17.917650,105.812452,26363.303778,1,0.03,0.250000,0.720000,11254.164413,0.0,0.020806


In [10]:
df_typhoon_names = df[['typhoon_name','typhoon_year']].drop_duplicates().reset_index(drop=True)

# .to_csv("data/typhoon_names_years.csv", index=False)

In [12]:
df_typhoon_names.to_csv("data/typhoon_names_years.csv", index=False)

In [13]:
df_2 = pd.read_csv("data/best_track_27ty.csv")

In [15]:
df_2 = df_2[['STORMNAME','YYYYMMDDHH','tyid']]

In [31]:
df_2

,STORMNAME,YYYYMMDDHH,tyid
0,DURIAN,2.006113e+11,18
1,DURIAN,2.006113e+11,18
2,DURIAN,2.006113e+11,18
3,DURIAN,2.006113e+11,18
4,DURIAN,2.006113e+11,18
...,...,...,...
1001,MANGKHUT,2.018092e+11,11
1002,MANGKHUT,2.018092e+11,11
1003,MANGKHUT,2.018092e+11,11
1004,MANGKHUT,2.018092e+11,11


In [56]:
# Sort by YYYYMMDDHH and keep first and last occurrence for each STORMNAME
df_sorted = df_2.sort_values('YYYYMMDDHH').reset_index(drop=True)

# Get first and last index for each STORMNAME
first_last_idx = (df_sorted.groupby('STORMNAME')
                  .apply(lambda x: [x.index[0], x.index[-1]])
                  .explode()
                  .unique())

result_df = df_sorted.loc[first_last_idx].sort_values('YYYYMMDDHH').reset_index(drop=True)

# print(result_df)
result_df['YYYYMMDDHH'] = pd.to_datetime(result_df['YYYYMMDDHH'].astype(int).astype(str), format='%Y%m%d%H%M')


In [58]:
result_df.to_csv("data/typhoon_names_years_2.csv", index=False)

In [60]:
df_new = pd.read_csv("data/typhoon_names_years_updated.csv")

In [66]:
starts = df_new.drop_duplicates(subset=['STORMNAME'], keep='first')
ends = df_new.drop_duplicates(subset=['STORMNAME'], keep='last')

ends.rename(columns={'YYYYMMDDHH': 'end_time'}, inplace=True)
starts.rename(columns={'YYYYMMDDHH': 'start_time'}, inplace=True)

joined_df = pd.merge(starts, ends, on='STORMNAME')

In [68]:
joined_df.drop(columns=['tyid_x', 'tyid_y'], inplace=True)

In [70]:
joined_df.to_csv("data/typhoon_dates.csv", index=False)

In [112]:
import numpy as np
import geopandas as gpd
from rasterio import transform
from rasterio.features import rasterize
import rasterio

# Load shapefile and reproject to WGS84 if needed
gdf = gpd.read_file("data/shape_files/buffered_files/PH_dissolved_buffer.shp")
gdf.set_crs("EPSG:32651", inplace=True)  # set CRS if not already set
gdf = gdf.to_crs("EPSG:4326")     # reproject to degrees

# Define resolution in degrees
resolution = 0.00416666666666667268

buffer = 0.1  # degrees, adjust as needed

minx, miny, maxx, maxy = gdf.total_bounds
minx -= buffer
miny -= buffer
maxx += buffer
maxy += buffer

width = int((maxx - minx) / resolution)
height = int((maxy - miny) / resolution)

out_transform = transform.from_bounds(minx, miny, maxx, maxy, width, height)

mask = rasterize(
    [(geom, 255) for geom in gdf.geometry],
    out_shape=(height, width),
    transform=out_transform,
    fill=0,
    dtype="uint8"
)

with rasterio.open(
    "data/philippines.tif",
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="uint8",
    crs=gdf.crs,
    transform=out_transform,
) as dst:
    dst.write(mask, 1)

print(f"Done! Dimensions: {width} x {height}")

Done! Dimensions: 3008 x 4018


In [73]:
print(gdf.crs)

None


In [80]:
print(f"Done! Dimensions: {width} x {height}")

Done! Dimensions: 3440 x 4450


In [94]:
tester = joined_df.iloc[:1,:]

In [ ]:
for event in tester.itertuples():
    print(event.Index,event.STORMNAME.lower(), event.start_time, event.end_time)
    print(pd.to_datetime(event.start_time).tz_localize(None))

0 durian 2006-11-25 6:00:00 2006-12-05 18:00:00
2006-11-25 06:00:00


In [100]:
joined_df = pd.read_csv("data/typhoon_dates.csv") #.iloc[:1,:]

In [101]:
joined_df

,STORMNAME,start_time,end_time
0,DURIAN,2006-11-25 6:00:00,2006-12-05 18:00:00
1,FENGSHEN,2008-06-17 18:00:00,2008-06-27 0:00:00
2,KETSANA,2009-09-25 0:00:00,2009-09-30 18:00:00
3,CONSON,2010-07-11 12:00:00,2010-07-18 0:00:00
4,NESAT,2011-09-23 0:00:00,2011-09-30 18:00:00
5,BOPHA,2012-11-25 18:00:00,2012-12-09 6:00:00
6,UTOR,2013-08-08 12:00:00,2013-08-18 6:00:00
7,TRAMI,2013-08-16 12:00:00,2013-08-24 6:00:00
8,USAGI,2013-09-16 0:00:00,2013-09-24 0:00:00
9,NARI,2013-10-08 12:00:00,2013-10-16 12:00:00


In [108]:
import xarray as xr

ds = xr.open_dataset("data/vectors/h24v06_2013.nc",engine="netcdf4")

In [109]:
ds.head()

<xarray.Dataset> Size: 2kB
Dimensions:                                    (time: 5, y: 5, x: 5)
Coordinates:
  * time                                       (time) datetime64[ns] 40B 2013...
  * y                                          (y) float64 40B 30.0 ... 29.98
  * x                                          (x) float64 40B 60.0 ... 60.02
Data variables: (12/13)
    ntl                                        (time, y, x) float32 500B ...
    ghsl_smod_2010                             (y, x) float32 100B ...
    ghsl_smod_2015                             (y, x) float32 100B ...
    ghsl_smod_2020                             (y, x) float32 100B ...
    ghsl_smod_2025                             (y, x) float32 100B ...
    ghs_smod_urban_suburban_2010               (y, x) float32 100B ...
    ...                                         ...
    ghs_smod_urban_suburban_2015               (y, x) float32 100B ...
    ghs_smod_urban_suburban_2015_2020combi     (y, x) float32 100B ...
    ghs_smod_urban_suburban_2020               (y, x) float32 100B ...
    ghs_smod_urban_suburban_2020_2025combi     (y, x) float32 100B ...
    ghs_smod_urban_suburban_2025               (y, x) float32 100B ...
    ghs_smod_urban_suburban_combined_mask_all  (y, x) float32 100B ...
Attributes:
    geotransform:  Affine(0.004166666666666673, 0.0, 60.0,\n       0.0, -0.00...
    GeoTransform:  60 0.0041666666666666727 0 30 0 -0.0041666666666666727
    crs_wkt:       GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137...

In [110]:
import matplotlib
matplotlib.use('Agg')
from osgeo import gdal
from osgeo import ogr  
import shapefile
ogr.UseExceptions()
import os
import random
import sys

gdal.PushErrorHandler('CPLQuietErrorHandler')

recalc = False
region = "philippines"

folder_base = "data"
polygon_folder = f"{folder_base}/shape_files/buffered_files/"
# ntlthesis/data/shape_files/buffered_files

# /PH_dissolved_buffer.shp
i=0

for filename in os.listdir(polygon_folder):
    print (filename)
    if filename.endswith(".shp"):
                        
        row=2400
        col=2400
        
        shp_file = f"{polygon_folder}/PH_dissolved_buffer.shp"

        ds_shp=shapefile.Reader(shp_file)  

        shapes = ds_shp.shapes()
        if not shapes:
            print(f"Shapefile has no shapes. Skipping.")
            continue

        try:
            minX, minY, maxX, maxY = shapes[0].bbox 
        except AttributeError:
            points = shapes[0].points
            if not points:
                continue
            xs, ys = zip(*points)
            minX, maxX = min(xs), max(xs)
            minY, maxY = min(ys), max(ys)
        
        h_min = int((minX + 180)//10)
        h_max = int((maxX + 180)//10)
        v_min = int((90 - maxY)//10)
        v_max = int((90 - minY)//10)
        
        # root_dir_NTL = "/gpfs/work4/0/FWC2/MYRIAD/data/NTL_data/Processed_data/Black_Marble_processed/corrected_NTL/2013/h00v01/" #contains NTL data
            
        # tifs = [f for f in os.listdir(root_dir_NTL)]
        # random_file = random.choice(tifs)
        # NTL_path = os.path.join(root_dir_NTL, random_file)
        # NTL_ds = gdal.Open(NTL_path)    
        im_proj = "EPSG:4326"

        #shp2raster
        raster_folder = f"{folder_base}/vectors/"
    
        output_path = os.path.join(raster_folder, f"{region}.tif")
        if os.path.exists(output_path) and not recalc:
            print (output_path)
            print ("already calculated")
            continue

        if not ((h_max < 0 or h_min > 35) or (v_max < 0 or v_min > 35)):

            h_min = max(0, min(h_min, 35))
            h_max = max(0, min(h_max, 35))
            v_min = max(0, min(v_min, 17))
            v_max = max(0, min(v_max, 17))
            deltah = h_max - h_min + 1
            deltav = v_max - v_min + 1
        
            h = h_min
            v = v_min

            left = -180 + h * 10
            top = 90 - v * 10

            im_row = int(deltav * row)
            im_col = int(deltah * col)
        
            res = 0.004166666666666673
            im_geotrans = [left, res, 0, top, 0, -res]

            if not(os.path.exists(raster_folder)):
                os.makedirs(raster_folder)

            ds_shp = ogr.Open(shp_file)
            shp_layer = ds_shp.GetLayer()

            target_ds = gdal.GetDriverByName('GTiff').Create(raster_folder+region+'.tif', xsize=im_col, ysize=im_row, bands=1, eType=gdal.GDT_Byte)
            target_ds.SetGeoTransform(im_geotrans)
            target_ds.SetProjection(im_proj)
            band = target_ds.GetRasterBand(1)
            band.SetNoDataValue(-9999)
            band.FlushCache()
            gdal.RasterizeLayer(target_ds,[1],shp_layer)                        
        else:
            print("out of boundary")

PH_dissolved_buffer.dbf
PH_dissolved_buffer.cpg
PH_dissolved_buffer.shp
out of boundary
PH_dissolved_buffer.qmd
PH_dissolved_buffer.shx


In [116]:
# ds_2012 = xr.open_dataset("data/vectors/h29v06_2012.nc",engine="netcdf4")
nc_ds = xr.open_dataset("data/h29v06_2013.nc",engine="netcdf4")


In [ ]:
# ds_2013
total_days_analysis: 322, rows_aff_chunk: 1323, cols_aff: 2960
stacked_NTL = np.full((322, 1323, 2960), np.nan)
t0 = 189
stack_row_start = 0
stack_row_end = 1323
stack_col_start = 0
stack_col_end = 2960
tile_row_start = 24
tile_row_end = 1347
tile_col_start = 24
tile_col_end = 2984
doy_start = 1
doy_end = 133

for i in range(1):

    max_time = int(nc_ds.sizes["time"])
    if doy_start - 1 >= max_time:
        continue
    avail_end = min(doy_end, max_time)
    print(f"DOY {doy_start} to {avail_end}")
    if avail_end < doy_start:
        continue
    t1_use = t0 + (avail_end - doy_start + 1)
    ntl_block = nc_ds["ntl"].isel(
        time=slice(doy_start - 1, avail_end),
        y=slice(tile_row_start, tile_row_end),
        x=slice(tile_col_start, tile_col_end),
    ).values
    stacked_NTL[
        t0:t1_use,
        stack_row_start:stack_row_end,
        stack_col_start:stack_col_end,
    ] = ntl_block

DOY 1 to 133


ValueError: could not broadcast input array from shape (133,1323,2376) into shape (133,1323,2960)

In [119]:
ntl_block.shape

(133, 1323, 2376)

In [2]:
hazard_tif_path = f"data/vectors/philippines.tif"
with rasterio.open(hazard_tif_path) as src:
    print(src.crs)
    print(src.transform)

EPSG:4326
| 0.00, 0.00, 114.17|
| 0.00,-0.00, 21.23|
| 0.00, 0.00, 1.00|


In [4]:
hazard_shp_path = f"data/shape_files/buffered_files/PH_dissolved_buffer.shp"
gdf_raw = gpd.read_file(hazard_shp_path)
print(gdf_raw.crs)
print(gdf_raw.total_bounds)

None
[-456585.30686949  507550.44838931  898641.02474252 2336462.6514199 ]


In [5]:
gdf = gpd.read_file(hazard_shp_path)
gdf = gdf.set_crs("EPSG:32651")
gdf = gdf.to_crs("EPSG:4326")
print(gdf.total_bounds)

[114.2737054    4.58311877 126.60914674  21.12606341]


In [27]:
with rasterio.open(hazard_tif_path) as src:
    print(repr(src.transform))
    # print(src.res)
    # print(src.transform)
    # print(src.width)
    # print(src.height)

Affine(0.004167367465979672, 0.0, 114.17370540324795,
       0.0, -0.004166984729694515, 21.22606341141669)


In [7]:
import xarray as xr
ds = xr.open_dataset("data/h29v06_2013.nc")  # or whatever a real tile path is
print(ds.sizes)
print(ds)

Frozen({'time': 365, 'y': 2400, 'x': 2400})
<xarray.Dataset> Size: 9GB
Dimensions:                                    (time: 365, y: 2400, x: 2400)
Coordinates:
  * time                                       (time) datetime64[ns] 3kB 2013...
  * y                                          (y) float64 19kB 30.0 ... 20.0
  * x                                          (x) float64 19kB 110.0 ... 120.0
Data variables: (12/13)
    ntl                                        (time, y, x) float32 8GB ...
    ghsl_smod_2010                             (y, x) float32 23MB ...
    ghsl_smod_2015                             (y, x) float32 23MB ...
    ghsl_smod_2020                             (y, x) float32 23MB ...
    ghsl_smod_2025                             (y, x) float32 23MB ...
    ghs_smod_urban_suburban_2010               (y, x) float32 23MB ...
    ...                                         ...
    ghs_smod_urban_suburban_2015               (y, x) float32 23MB ...
    ghs_smod_urban_sub

In [15]:
def split_analysis_region(xmax, xmin, chunk_num=3):
            '''
            Split the analysis region into chunks for processing. The processing area was
            too large for the philippines in the `if estimated_gb > MEM_thresh:` check, 
            so this function can be used to split the area into smaller chunks that are processed 
            sequentially. The results can then be combined at the end.
            '''
            row_range = xmax - xmin
            chunk_size = row_range // chunk_num

            return [(xmin + i * chunk_size, xmin + (i + 1) * chunk_size) for i in range(chunk_num)]

In [13]:
xmin = 0
xmax = 2400 
chunk_size = 5

row_sizes = (xmax - xmin) // chunk_size

bounds = [(xmin + i * row_sizes, xmin + (i + 1) * row_sizes) for i in range(chunk_size)]

In [14]:
bounds

[(0, 480), (480, 960), (960, 1440), (1440, 1920), (1920, 2400)]

In [21]:
import json

# Open the JSON file and load its content
with open('metrics_dict_debug.json', 'r') as file:
    data = json.load(file)

# Now 'data' is a Python dictionary or list
print(data)

{'output_base': {'available_percentage': {'chunk_files': ['data/output//temp/available_percentage/chunk1_philippines_bopha_0_available.tif', 'data/output//temp/available_percentage/chunk2_philippines_bopha_0_available.tif', 'data/output//temp/available_percentage/chunk3_philippines_bopha_0_available.tif'], 'final_path': 'data/output//available_percentage/philippines_bopha_0_available.tif'}, 'pre_std': {'chunk_files': ['data/output//temp/pre_std/chunk1_philippines_bopha_0_pre_event_std.tif', 'data/output//temp/pre_std/chunk2_philippines_bopha_0_pre_event_std.tif', 'data/output//temp/pre_std/chunk3_philippines_bopha_0_pre_event_std.tif'], 'final_path': 'data/output//pre_std/philippines_bopha_0_pre_event_std.tif'}, 'pre_mean': {'chunk_files': ['data/output//temp/pre_mean/chunk1_philippines_bopha_0_pre_event_mean.tif', 'data/output//temp/pre_mean/chunk2_philippines_bopha_0_pre_event_mean.tif', 'data/output//temp/pre_mean/chunk3_philippines_bopha_0_pre_event_mean.tif'], 'final_path': 'data/

In [18]:
data['output_base']['affected']['chunk_files']

['data/output//temp/affected/chunk1_philippines_bopha_0_aff.tif',
 'data/output//temp/affected/chunk2_philippines_bopha_0_aff.tif',
 'data/output//temp/affected/chunk3_philippines_bopha_0_aff.tif']

In [ ]:


    with rasterio.open(f) as src:
        print(f)
        print(f"  transform: {src.transform}")
        print(f"  bounds: {src.bounds}")
        print(f"  shape: {src.width}, {src.height}")

data/output//temp/affected/chunk1_philippines_bopha_0_aff.tif
  transform: | 0.00, 0.00, 114.17|
| 0.00,-0.00, 21.23|
| 0.00, 0.00, 1.00|
  bounds: BoundingBox(left=114.17370540324795, bottom=4.483118767504127, right=126.7091467409148, top=21.22606341141669)
  shape: 3008, 4018
data/output//temp/affected/chunk2_philippines_bopha_0_aff.tif
  transform: | 0.00, 0.00, 114.17|
| 0.00,-0.00, 21.23|
| 0.00, 0.00, 1.00|
  bounds: BoundingBox(left=114.17370540324795, bottom=4.483118767504127, right=126.7091467409148, top=21.22606341141669)
  shape: 3008, 4018
data/output//temp/affected/chunk3_philippines_bopha_0_aff.tif
  transform: | 0.00, 0.00, 114.17|
| 0.00,-0.00, 21.23|
| 0.00, 0.00, 1.00|
  bounds: BoundingBox(left=114.17370540324795, bottom=4.483118767504127, right=126.7091467409148, top=21.22606341141669)
  shape: 3008, 4018


In [22]:
for f in data["output_base"]["affected"]["chunk_files"]:
    with rasterio.open(f) as src:
        print(f"transform: {src.transform}")
        print(f"bounds: {src.bounds}")
        print(f"shape: {src.width}, {src.height}")

transform: | 0.00, 0.00, 114.17|
| 0.00,-0.00, 21.23|
| 0.00, 0.00, 1.00|
bounds: BoundingBox(left=114.17370540324795, bottom=4.483118767504127, right=126.7091467409148, top=21.22606341141669)
shape: 3008, 4018
transform: | 0.00, 0.00, 114.17|
| 0.00,-0.00, 21.23|
| 0.00, 0.00, 1.00|
bounds: BoundingBox(left=114.17370540324795, bottom=4.483118767504127, right=126.7091467409148, top=21.22606341141669)
shape: 3008, 4018
transform: | 0.00, 0.00, 114.17|
| 0.00,-0.00, 21.23|
| 0.00, 0.00, 1.00|
bounds: BoundingBox(left=114.17370540324795, bottom=4.483118767504127, right=126.7091467409148, top=21.22606341141669)
shape: 3008, 4018


In [23]:
import xarray as xr
ds = xr.open_dataset("data/h29v06_2013.nc")
print(f"NC geotransform: {ds.attrs.get('GeoTransform')}")
print(f"x range: {float(ds.x.min())} to {float(ds.x.max())}")
print(f"y range: {float(ds.y.min())} to {float(ds.y.max())}")

NC geotransform: 110 0.0041666666666666727 0 30 0 -0.0041666666666666727
x range: 110.00208333333333 to 119.99791666666668
y range: 20.002083333333317 to 29.997916666666665


In [24]:
ds7 = xr.open_dataset("data/h29v07_2013.nc")
print(f"h29v07 x range: {float(ds7.x.min())} to {float(ds7.x.max())}")
print(f"h29v07 y range: {float(ds7.y.min())} to {float(ds7.y.max())}")

ds8 = xr.open_dataset("data/h29v08_2013.nc")
print(f"h29v08 x range: {float(ds8.x.min())} to {float(ds8.x.max())}")
print(f"h29v08 y range: {float(ds8.y.min())} to {float(ds8.y.max())}")

h29v07 x range: 110.00208333333333 to 119.99791666666668
h29v07 y range: 10.002083333333317 to 19.997916666666665
h29v08 x range: 110.00208333333333 to 119.99791666666668
h29v08 y range: 0.002083333333319004 to 9.997916666666667


NameError: name 'haz_tif' is not defined

In [29]:
import matplotlib.pyplot as plt
import rasterio
import geopandas as gpd
import numpy as np

BUFFER_pre = 150 
BUFFER_post = 150
NODATA_VALUE = -9999
RECOVERY_NOT_FOUND = 999
NOT_ENOUGH_DATA = 888
# NEW: availability threshold for postimpact/postworst and availability-adjusted durations
AVAIL_THRESHOLDS = [0.3, 0.5]

# Load raster
with rasterio.open("data/output/affected/philippines_bopha_0_aff.tif") as src:
    data = src.read(1)
    bounds = src.bounds

# Load and reproject shapefile
gdf = gpd.read_file(hazard_shp_path)
gdf = gdf.set_crs("EPSG:32651")
gdf = gdf.to_crs("EPSG:4326")

fig, ax = plt.subplots(figsize=(10, 10))

# Plot raster
ax.imshow(np.where(data == NODATA_VALUE, np.nan, data), 
          extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
          cmap='Greys', origin='upper')

# Plot shapefile
gdf.boundary.plot(ax=ax, color='red', linewidth=1)

plt.title("Raster vs Shapefile")
plt.savefig("debug_plot.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved to debug_plot.png")

Saved to debug_plot.png


/var/folders/xw/1pm761sd11b96_3c77wx1b900000gn/T/ipykernel_69388/4270026532.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [48]:
pd.read_csv(f"data/typhoon_dates.csv").iloc[1:2,:] #just the first event for testing

,STORMNAME,start_time,end_time
1,UTOR,2013-08-08 12:00:00,2013-08-18 6:00:00


In [57]:
pd.read_csv(f"data/typhoon_dates.csv").iloc[21:,:]

,STORMNAME,start_time,end_time
21,MANGKHUT,2018-09-07 6:00:00,2018-09-17 0:00:00
22,KAMMURI,2019-11-30 23:00:00,2019-12-05 8:00:00
23,PHANFONE,2019-12-24 16:45:00,2019-12-28 0:00:00


In [41]:
events = pd.read_csv(f"data/typhoon_dates.csv").iloc[:1,:] #just the first event for testing

for event in events.itertuples():
    print(event.start_time, event.end_time)

2012-11-25 18:00:00 2012-12-09 6:00:00


In [ ]:
import subprocess

files = [
    "file1.nc",
    "file2.nc",
    "file3.nc",
]

years = ['2013', '2014', '2015', '2016', '2017', '2018', '2019']
tiles = ['h29v06', 'h29v07', 'h29v08', 'h30v06', 'h30v07', 'h30v08']

# files = [f"{tile}_{year}.nc" for tile in tiles for year in years]

server = "esegrestmorais@snellius.surf.nl:/gpfs/work4/0/FWC2/MYRIAD/data/NTL_data/Processed_data/Black_Marble_processed/corrected_NTL/"
local_destination = "/data/tiles/"

# Prompts once, hides input like a password field
password = "esmsndata2026"

for y in years:
    for t in tiles:
        file = f"{t}_{y}.nc"
        file_path = f"{server}{y}/{file}"
        print(f"Downloading {file}...")
        result = subprocess.run(
            ["sshpass", "-p", password, "scp", file_path, local_destination]
        )
        if result.returncode == 0:
            print(f"✓ {file}")
        else:
            print(f"✗ Failed: {file}")

FileNotFoundError: [Errno 2] No such file or directory: 'sshpass'

In [36]:
files

['h29v06_2013.nc',
 'h29v06_2014.nc',
 'h29v06_2015.nc',
 'h29v06_2016.nc',
 'h29v06_2017.nc',
 'h29v06_2018.nc',
 'h29v06_2019.nc',
 'h29v07_2013.nc',
 'h29v07_2014.nc',
 'h29v07_2015.nc',
 'h29v07_2016.nc',
 'h29v07_2017.nc',
 'h29v07_2018.nc',
 'h29v07_2019.nc',
 'h29v08_2013.nc',
 'h29v08_2014.nc',
 'h29v08_2015.nc',
 'h29v08_2016.nc',
 'h29v08_2017.nc',
 'h29v08_2018.nc',
 'h29v08_2019.nc',
 'h30v06_2013.nc',
 'h30v06_2014.nc',
 'h30v06_2015.nc',
 'h30v06_2016.nc',
 'h30v06_2017.nc',
 'h30v06_2018.nc',
 'h30v06_2019.nc',
 'h30v07_2013.nc',
 'h30v07_2014.nc',
 'h30v07_2015.nc',
 'h30v07_2016.nc',
 'h30v07_2017.nc',
 'h30v07_2018.nc',
 'h30v07_2019.nc',
 'h30v08_2013.nc',
 'h30v08_2014.nc',
 'h30v08_2015.nc',
 'h30v08_2016.nc',
 'h30v08_2017.nc',
 'h30v08_2018.nc',
 'h30v08_2019.nc']

In [42]:
def split_analysis_region(xmax, xmin, chunk_num=3):
            '''
            Split the analysis region into chunks for processing. The processing area was
            too large for the philippines in the `if estimated_gb > MEM_thresh:` check, 
            so this function can be used to split the area into smaller chunks that are processed 
            sequentially. The results can then be combined at the end.
            '''
            row_range = xmax - xmin
            chunk_size = row_range // chunk_num

            return [(xmin + i * chunk_size, xmin + (i + 1) * chunk_size) for i in range(chunk_num)]

In [45]:
split_analysis_region(0, 99, chunk_num=1)

[(99, 0)]

In [3]:
import xarray as xr
import numpy as np
import rioxarray
import rasterio
from rasterio.merge import merge
from rasterio.crs import CRS
import tempfile, os
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

#confusion matrix cmap
same_scale_color_map = mcolors.LinearSegmentedColormap.from_list(
    "my_cmap",
    ['#D6F1FF', '#33BBFF', '#0088ce', '#005F8F', '#001B29']
)

tiles = ["h30v08", "h30v07", "h30v06", "h29v08", "h29v07", "h29v06"]
year  = 2018
doy   = 300

# ── Write each tile to temp TIF and merge ────────────────────────────────────
tmp_files = []
for tile in tiles:
    nc_path = f"../../data/tiles/{year}/{tile}_{year}.nc"
    ds      = xr.open_dataset(nc_path)
    da      = ds["ntl"].isel(time=doy - 1)
    da      = da.rio.set_spatial_dims(x_dim="x", y_dim="y").rio.write_crs("EPSG:4326")
    tmp     = tempfile.NamedTemporaryFile(suffix=".tif", delete=False)
    da.rio.to_raster(tmp.name)
    tmp_files.append(tmp.name)
    print(f"Written tile {tile}")

src_files         = [rasterio.open(f) for f in tmp_files]
mosaic, transform = merge(src_files)

# ── Visualise (Black Marble style) ───────────────────────────────────────────
img     = mosaic[0].copy().astype(float)
img[img <= 0]     = np.nan
img[img == -9999] = np.nan

img_log = np.log1p(img)
vmin    = np.nanpercentile(img_log, 0)
vmax    = np.nanpercentile(img_log, 99.9)

fig, ax = plt.subplots(figsize=(14, 12), facecolor='black')
ax.imshow(img_log, cmap=same_scale_color_map, vmin=vmin, vmax=vmax)
ax.set_facecolor('black')
ax.set_axis_off()
plt.tight_layout(pad=0)
plt.savefig(f"ntl_black_marble_{year}_doy{doy}.png", dpi=200,
            bbox_inches="tight", facecolor='black')
plt.close()
print("Saved visualisation")

# ── Save merged TIF ──────────────────────────────────────────────────────────
profile = src_files[0].profile.copy()
profile.update({
    "driver":    "GTiff",
    "height":    mosaic.shape[1],
    "width":     mosaic.shape[2],
    "transform": transform,
    "crs":       CRS.from_epsg(4326),
    "dtype":     "float64",
    "compress":  "lzw",
    "count":     1,
})

output_path = f"data/output_new/ntl_mosaic_{year}_doy{doy}.tif"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with rasterio.open(output_path, "w", **profile) as dst:
    dst.write(mosaic[0], 1)
print(f"Saved TIF: {output_path}")

# ── Cleanup ───────────────────────────────────────────────────────────────────
for src in src_files:
    src.close()
for f in tmp_files:
    os.unlink(f)

Written tile h30v08
Written tile h30v07
Written tile h30v06
Written tile h29v08
Written tile h29v07
Written tile h29v06
Saved visualisation
Saved TIF: data/output_new/ntl_mosaic_2018_doy300.tif


In [6]:
# ML
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import (
    BaggingRegressor, ExtraTreesRegressor, GradientBoostingRegressor,
    HistGradientBoostingRegressor, RandomForestRegressor,
)
from sklearn.exceptions import DataConversionWarning
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV, RepeatedKFold,
    cross_val_score, train_test_split,
)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVR

def get_na_columns(df):
    nas_finder_new = df.isna().sum()
    return nas_finder_new[nas_finder_new > 0].keys()

# data cleaning functions
def find_bad_storms(df, na_columns):
    # find storms where ALL values are null for any of the cols
    bad_storms = []
    for storm in df['typhoon_name'].unique():
        for col in na_columns:
            if df[df['typhoon_name'] == storm][col].isna().all():
                bad_storms.append(storm)
                break  # no need to check other cols

    bad_storms = list(set(bad_storms))
    return bad_storms

# Load and split data
df = pd.read_csv('../../data/target/model_training_data_final.csv')


# get data
y = df['percent_houses_damaged']
X = df.drop(columns=['percent_houses_damaged_5years', 'percent_houses_damaged']) 

#get nulls
nas_finder = df.isna().sum()
too_null_cols = []
for i in nas_finder[nas_finder > 0].index: 
    if nas_finder[nas_finder > 0].loc[i]/df.shape[0] > 0.95:
        too_null_cols.append(i)

X.drop(columns=too_null_cols,inplace=True)

# remove storma that have no values
na_cols = get_na_columns(X)
bad_storms = find_bad_storms(X, na_cols)
mask = ~X['typhoon_name'].isin(bad_storms)
X = X[mask]
y = y[mask]

# drop columns that only have one value and therefore don't provide value
cols_to_drop = []
for col in X.columns:
    if X[col].value_counts().shape[0] == 1:
        cols_to_drop.append(col)

X.drop(columns=cols_to_drop, inplace=True)
X.drop(columns=['typhoon_name'], inplace=True)

#need to train-test split to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

print("for LGBM:")
lgb_model = lgb.LGBMRegressor(random_state=42, n_jobs=-1,verbose=0);
lgb_model.fit(X_train, y_train)
scores = cross_val_score(lgb_model, X_train, y_train, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")
y_pred_lgb = lgb_model.predict(X_test)
test_r2_lgb = r2_score(y_test, y_pred_lgb)
print(f"Test Set R²: {test_r2_lgb:.4f}")
print("for HistGradientBoostingRegressor:")
hgbr_model = HistGradientBoostingRegressor(random_state=42,verbose=0)
hgbr_model.fit(X_train, y_train)
scores = cross_val_score(hgbr_model, X_train, y_train, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")
y_pred_hgbr = hgbr_model.predict(X_test)
test_r2_hgbr = r2_score(y_test, y_pred_hgbr)
print(f"Test Set R²: {test_r2_hgbr:.4f}")
print("for BaggingRegressor:")
br_model = BaggingRegressor(random_state=42, n_jobs=-1, verbose=0)
br_model.fit(X_train, y_train)
scores = cross_val_score(br_model, X_train, y_train, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")
y_pred_br = br_model.predict(X_test)
test_r2_br = r2_score(y_test, y_pred_br)
print(f"Test Set R²: {test_r2_br:.4f}")
print("for XGBRegressor:")
xgb_b_model = xgb.XGBRegressor(random_state=42)
xgb_b_model.fit(X_train, y_train)
scores = cross_val_score(xgb_b_model, X_train, y_train, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")
y_pred_xgbb = xgb_b_model.predict(X_test)
test_r2_xgbb = r2_score(y_test, y_pred_xgbb)
print(f"Test Set R²: {test_r2_xgbb:.4f}")

for LGBM:
mean CV score: 0.1313
Test Set R²: 0.2281
for HistGradientBoostingRegressor:
mean CV score: 0.1280
Test Set R²: 0.2023
for BaggingRegressor:
mean CV score: -0.0805
Test Set R²: 0.0529
for XGBRegressor:
mean CV score: 0.0824
Test Set R²: 0.1254


In [8]:
#need to train-test split to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
X_train.drop(columns=['grid_point_id'],inplace=True)
X_test.drop(columns=['grid_point_id'],inplace=True)

print("for LGBM:")
lgb_model = lgb.LGBMRegressor(random_state=42, n_jobs=-1,verbose=0);
lgb_model.fit(X_train, y_train)
scores = cross_val_score(lgb_model, X_train, y_train, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")
y_pred_lgb = lgb_model.predict(X_test)
test_r2_lgb = r2_score(y_test, y_pred_lgb)
print(f"Test Set R²: {test_r2_lgb:.4f}")
print("for HistGradientBoostingRegressor:")
hgbr_model = HistGradientBoostingRegressor(random_state=42,verbose=0)
hgbr_model.fit(X_train, y_train)
scores = cross_val_score(hgbr_model, X_train, y_train, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")
y_pred_hgbr = hgbr_model.predict(X_test)
test_r2_hgbr = r2_score(y_test, y_pred_hgbr)
print(f"Test Set R²: {test_r2_hgbr:.4f}")
print("for BaggingRegressor:")
br_model = BaggingRegressor(random_state=42, n_jobs=-1, verbose=0)
br_model.fit(X_train, y_train)
scores = cross_val_score(br_model, X_train, y_train, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")
y_pred_br = br_model.predict(X_test)
test_r2_br = r2_score(y_test, y_pred_br)
print(f"Test Set R²: {test_r2_br:.4f}")
print("for XGBRegressor:")
xgb_b_model = xgb.XGBRegressor(random_state=42)
xgb_b_model.fit(X_train, y_train)
scores = cross_val_score(xgb_b_model, X_train, y_train, cv=5, scoring='r2')
print(f"mean CV score: {scores.mean():.4f}")
y_pred_xgbb = xgb_b_model.predict(X_test)
test_r2_xgbb = r2_score(y_test, y_pred_xgbb)
print(f"Test Set R²: {test_r2_xgbb:.4f}")

for LGBM:
mean CV score: 0.0152
Test Set R²: 0.0402
for HistGradientBoostingRegressor:
mean CV score: 0.0248
Test Set R²: 0.0463
for BaggingRegressor:
mean CV score: -0.1281
Test Set R²: -0.1525
for XGBRegressor:
mean CV score: -0.0875
Test Set R²: -0.0580
